<a href="https://colab.research.google.com/github/Omar-Condori/AYMA-ARUX/blob/main/notebooks/configuracion_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AYMA-ARUX — Speech-to-Speech Translation
## ASR (Whisper + LoRA) → MT (NLLB-200) → TTS (MMS)

## 1. Clonar repositorio

In [ ]:
%cd /content/
!rm -rf AYMA-ARUX
!git clone https://github.com/Omar-Condori/AYMA-ARUX.git
%cd /content/AYMA-ARUX
!ls

## 2. Instalar dependencias

In [ ]:
%cd /content/
!rm -rf AYMA-ARUX
!git clone https://github.com/Omar-Condori/AYMA-ARUX.git
%cd /content/AYMA-ARUX
!ls

## 3. Verificar GPU

In [ ]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ Sin GPU — activar T4 en Editar > Configuración de aceleración")

## 4. Conectar Google Drive y copiar datos

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/AYMA-ARUX-resultados"
os.makedirs(DRIVE_DATA, exist_ok=True)

print("✅ Drive montado")

In [ ]:
# Copiar audios desde Drive a Colab
ORIGEN_AUDIOS = f"{DRIVE_DATA}/Dataset_Audios_Tesis"
DESTINO_AUDIOS = "/content/AYMA-ARUX/datos/raw/audio"

if os.path.exists(ORIGEN_AUDIOS):
    os.makedirs(DESTINO_AUDIOS, exist_ok=True)
    for f in os.listdir(ORIGEN_AUDIOS):
        if f.endswith(".wav"):
            shutil.copy2(os.path.join(ORIGEN_AUDIOS, f), os.path.join(DESTINO_AUDIOS, f))
    print(f"✅ Audios copiados: {len(os.listdir(DESTINO_AUDIOS))} archivos")
else:
    print(f"❌ No se encontró la carpeta {ORIGEN_AUDIOS}")
    print("   Sube tus audios a Drive en: AYMA-ARUX-resultados/Dataset_Audios_Tesis/")

In [ ]:
# Usar CSV del repositorio. Si Drive tiene uno con formato correcto, usarlo
import pandas as pd

REPO_CSV = "/content/AYMA-ARUX/datos/datos_corpus.csv"
DRIVE_CSV = f"{DRIVE_DATA}/datos_corpus.csv"

usar_drive = False
if os.path.exists(DRIVE_CSV):
    df_drive = pd.read_csv(DRIVE_CSV)
    if "archivo_wav" in df_drive.columns and len(df_drive) > 0:
        if str(df_drive["archivo_wav"].iloc[0]).startswith("SPK000"):
            shutil.copy2(DRIVE_CSV, REPO_CSV)
            usar_drive = True

df = pd.read_csv(REPO_CSV)
nombre = "Drive" if usar_drive else "Repositorio"
print(f"✅ Usando CSV de {nombre} ({len(df)} filas, ej: {df['archivo_wav'].iloc[0]})")

## 5. ETL — Limpiar audios y texto

In [ ]:
import sys
sys.path.insert(0, "/content/AYMA-ARUX")
!python src/utils/etl_limpiar_datos.py

## 6. Verificar datos limpios

In [ ]:
import pandas as pd
import os

df = pd.read_csv("/content/AYMA-ARUX/datos/processed/metadata_clean.csv")
print(f"Filas en CSV limpio: {len(df)}")

audios = [f for f in os.listdir("/content/AYMA-ARUX/datos/processed/audio") if f.endswith(".wav")]
print(f"Audios procesados: {len(audios)}")

archivos_csv = set(df["archivo_wav"])
archivos_dir = set(audios)
coinciden = archivos_csv & archivos_dir
print(f"Coinciden: {len(coinciden)}/{len(df)}")

total_seg = df["duracion_seg"].sum()
print(f"Duración total: {total_seg:.2f}s ({total_seg/60:.1f} min)")

if len(coinciden) == len(df):
    print("✅ Datos limpios — listo para entrenar")
else:
    faltan = archivos_csv - archivos_dir
    print(f"❌ Faltan {len(faltan)} audios: {list(faltan)[:5]}...")

## 7. Crear dataset HuggingFace

In [ ]:
import sys
sys.path.insert(0, "/content/AYMA-ARUX")

# Ejecutar creación de dataset
!python src/utils/crear_dataset_hf.py

## 8. Entrenar ASR con LoRA

> ⏱ Tiempo estimado con 110 audios en T4: **~10-15 minutos**

In [ ]:
# Entrenar Whisper + LoRA
!python src/services/entrenar_lora_whisper.py

## 9. Guardar modelo en Drive

In [ ]:
import shutil

MODELO_ORIGEN = "/content/AYMA-ARUX/modelos/asr/final"
MODELO_DESTINO = f"{DRIVE_DATA}/modelos/asr/final"

os.makedirs(MODELO_DESTINO, exist_ok=True)
for item in os.listdir(MODELO_ORIGEN):
    s = os.path.join(MODELO_ORIGEN, item)
    d = os.path.join(MODELO_DESTINO, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)
print(f"✅ Modelo guardado en Drive: {MODELO_DESTINO}")

## 10. Probar pipeline completo

In [ ]:
import os
os.makedirs("/content/AYMA-ARUX/resultados/pipeline", exist_ok=True)

# Probar con un audio limpio de ejemplo
audio_prueba = "/content/AYMA-ARUX/datos/processed/audio/SPK00001_00001.wav"
audio_salida = "/content/AYMA-ARUX/resultados/pipeline/resultado.wav"

!python -c """
import sys
sys.path.insert(0, '/content/AYMA-ARUX')
from src.pipeline.pipeline_completo import voz_a_voz
voz_a_voz('{audio_prueba}', '{audio_salida}', direccion='aym-spa')
"""

In [ ]:
# Copiar resultados a Drive
RESULTADOS_ORIGEN = "/content/AYMA-ARUX/resultados"
RESULTADOS_DESTINO = f"{DRIVE_DATA}/resultados"

os.makedirs(RESULTADOS_DESTINO, exist_ok=True)
for item in os.listdir(RESULTADOS_ORIGEN):
    s = os.path.join(RESULTADOS_ORIGEN, item)
    d = os.path.join(RESULTADOS_DESTINO, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)
print(f"✅ Resultados guardados en Drive: {RESULTADOS_DESTINO}")

## ✅ Listo

El modelo entrenado y los resultados están en tu Google Drive.